# 01 — Data Pipeline & Feature Engineering
Loads Tardis BTCUSDT trade data, builds bars, computes all 73 features.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from config import *
from utils import (
    BarBuilder, compute_tofi, compute_tofi_divergence,
    compute_basis_features, compute_funding_features,
    compute_realized_moments, compute_microstructure_features
)

DATA_DIR = Path('../data')
DATA_DIR.mkdir(exist_ok=True)

## Load Tardis Trade Data

In [ ]:
# Tardis exports CSV with columns: exchange, symbol, timestamp, id, side, price, amount
# side derived from is_buyer_maker: if is_buyer_maker=True -> side='sell', else -> side='buy'
# Adjust paths to your Tardis export location

SPOT_PATH = DATA_DIR / 'tardis_btcusdt_spot_trades.csv'
PERP_PATH = DATA_DIR / 'tardis_btcusdt_perp_trades.csv'

def load_tardis_trades(path: Path) -> pd.DataFrame:
    """Load Tardis trade CSV and normalize columns."""
    df = pd.read_csv(path)
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='us', utc=True)
    # Tardis uses is_buyer_maker; convert to taker side
    if 'is_buyer_maker' in df.columns:
        df['side'] = np.where(df['is_buyer_maker'], 'sell', 'buy')
    df = df[['timestamp', 'price', 'amount', 'side']].sort_values('timestamp')
    return df.reset_index(drop=True)

spot_trades = load_tardis_trades(SPOT_PATH)
perp_trades = load_tardis_trades(PERP_PATH)
print(f'Spot trades: {len(spot_trades):,}  |  Perp trades: {len(perp_trades):,}')
print(f'Date range: {spot_trades["timestamp"].min()} to {spot_trades["timestamp"].max()}')

## Build Bars

In [ ]:
builder = BarBuilder()

# 1-minute bars for features
spot_1m = builder.build_bars(spot_trades, bar_sec=60)
perp_1m = builder.build_bars(perp_trades, bar_sec=60)

# 1-second bars for realized moments
spot_1s = builder.build_bars(spot_trades, bar_sec=1)

# Align perp and spot on same index
common_idx = spot_1m.index.intersection(perp_1m.index)
spot_1m = spot_1m.loc[common_idx]
perp_1m = perp_1m.loc[common_idx]

print(f'1-min bars: {len(spot_1m):,}  |  1-sec bars: {len(spot_1s):,}')

## Compute Directional Features

In [ ]:
# TOFI — 9 variants
spot_tofi = compute_tofi(spot_1m, prefix='spot_')
perp_tofi = compute_tofi(perp_1m, prefix='perp_')
tofi_div = compute_tofi_divergence(spot_tofi, perp_tofi).rename('tofi_perp_spot_divergence')

# Cross-exchange basis — 6 variants
basis = compute_basis_features(perp_1m, spot_1m)

print(f'TOFI features: {spot_tofi.shape[1] + perp_tofi.shape[1] + 1}')
print(f'Basis features: {basis.shape[1]}')

In [ ]:
# Funding rate & OI features — 7 variants
# These require additional Tardis data channels (funding_rate, open_interest, liquidations)
# Placeholder: load if available, otherwise create empty

FUNDING_PATH = DATA_DIR / 'tardis_btcusdt_funding.csv'
OI_PATH = DATA_DIR / 'tardis_btcusdt_oi.csv'
LIQ_PATH = DATA_DIR / 'tardis_btcusdt_liquidations.csv'

funding_rate = pd.Series(dtype=float, name='funding_rate')
oi = pd.Series(dtype=float, name='open_interest')
liquidations = pd.DataFrame()

if FUNDING_PATH.exists():
    fr_df = pd.read_csv(FUNDING_PATH)
    fr_df['timestamp'] = pd.to_datetime(fr_df['timestamp'], unit='us', utc=True)
    fr_df = fr_df.set_index('timestamp').resample('1min').last()
    funding_rate = fr_df['funding_rate'].reindex(spot_1m.index, method='ffill')

if OI_PATH.exists():
    oi_df = pd.read_csv(OI_PATH)
    oi_df['timestamp'] = pd.to_datetime(oi_df['timestamp'], unit='us', utc=True)
    oi_df = oi_df.set_index('timestamp').resample('1min').last()
    oi = oi_df['open_interest'].reindex(spot_1m.index, method='ffill')

funding_feats = compute_funding_features(
    funding_rate.reindex(spot_1m.index, fill_value=0),
    oi.reindex(spot_1m.index, fill_value=0),
    liquidations if len(liquidations) > 0 else None,
    perp_tofi.get('perp_tofi_5m')
)
print(f'Funding features: {funding_feats.shape[1]}')

In [ ]:
# Realized higher moments — 6 variants
moments = compute_realized_moments(spot_1s)
# Resample to 1-min for joining
moments_1m = moments.resample('1min').last().reindex(spot_1m.index, method='ffill')
print(f'Moment features: {moments_1m.shape[1]}')

In [ ]:
# Microstructure features (HAR-RV, VPIN, Kyle's Lambda, etc.) — ~44 variants
micro = compute_microstructure_features(spot_1m)
print(f'Microstructure features: {micro.shape[1]}')

## Assemble Feature Matrix

In [ ]:
features = pd.concat([
    spot_tofi, perp_tofi, tofi_div,
    basis,
    funding_feats,
    moments_1m,
    micro,
], axis=1)

# Add price columns for downstream use
features['spot_close'] = spot_1m['close']
features['perp_close'] = perp_1m['close']
features['log_return_1m'] = np.log(spot_1m['close'] / spot_1m['close'].shift(1))

# 15-min forward binary target: did price go up?
features['target_up'] = (spot_1m['close'].shift(-15) > spot_1m['close']).astype(float)

print(f'Total features: {features.shape[1] - 3}')  # exclude price cols and target
print(f'Rows: {len(features):,}')

In [ ]:
# Save
features.to_parquet(DATA_DIR / 'features.parquet')
print(f'Saved to {DATA_DIR / "features.parquet"}')

## Validation

In [ ]:
# NaN rates by feature group
feat_cols = [c for c in features.columns if c not in ['spot_close', 'perp_close', 'target_up']]
nan_rates = features[feat_cols].isna().mean().sort_values(ascending=False)
print('Top NaN rates:')
print(nan_rates.head(15).to_string())
print(f'\nFeatures with >50% NaN: {(nan_rates > 0.5).sum()}')
print(f'Features with <5% NaN:  {(nan_rates < 0.05).sum()}')

In [ ]:
# Correlation heatmap of directional features
directional_cols = [c for c in features.columns if any(
    k in c for k in ['tofi', 'basis', 'funding', 'skew']
)]
if len(directional_cols) > 2:
    corr = features[directional_cols].corr()
    fig, ax = plt.subplots(figsize=(10, 8))
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
    ax.set_xticks(range(len(directional_cols)))
    ax.set_yticks(range(len(directional_cols)))
    ax.set_xticklabels(directional_cols, rotation=90, fontsize=7)
    ax.set_yticklabels(directional_cols, fontsize=7)
    plt.colorbar(im)
    plt.title('Directional Feature Correlations')
    plt.tight_layout()
    plt.show()